
# 01 — FakeMusicCaps Data Exploration

**Goal:** Explore the FakeMusicCaps dataset, examine its metadata and audio characteristics, and identify potential data quality issues before feature extraction.

**Dataset:** FakeMusicCaps — AI-generated music from five text-to-music generators.

This notebook covers:
- Dataset structure and generator distribution
- Metadata fields and prompt relationships
- Missing values and duplicate checks
- Technical audio characteristics
- Preprocessing decisions


### Data Setup

Before running this notebook, download the FakeMusicCaps audio dataset and MusicCaps metadata using the provided script.

From the project root directory, run:

```bash
python scripts/download_data.py
```

The script downloads the FakeMusicCaps archive and MusicCaps metadata (`musiccaps-public.csv`) into `data/raw/`. Existing downloaded files are reused. The FakeMusicCaps archive must also be extracted into `data/raw/fakemusiccaps/`, with the five generator directories inside it.

This notebook assumes that the required data is available locally and does not download it automatically.

If `soundfile` is not already installed run next block

In [ ]:
%pip install soundfile

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf


In [2]:
DATA_DIR = Path("../data/raw/fakemusiccaps")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}. Run scripts/download_data.py first.")

## 1. Dataset Structure

The FakeMusicCaps dataset contains 27,605 audio files generated by five text-to-music models: AudioLDM2, MusicGen Medium, MusicLDM, Mustango, and Stable Audio Open. Each generator contributes 5,521 audio files, resulting in a balanced dataset in terms of generator classes.

The extracted archive also contains a `__MACOSX` directory with auxiliary files, which was excluded from the analysis.

In [4]:
generator_dirs = sorted(folder for folder in DATA_DIR.iterdir() if folder.is_dir() and folder.name != "__MACOSX")
generator_counts = pd.DataFrame([{"generator": folder.name, "num_files": len(list(folder.glob("*.wav")))} for folder in generator_dirs])
display(generator_counts)
print("Total files:", generator_counts["num_files"].sum())

,generator,num_files
0,audioldm2,5521
1,MusicGen_medium,5521
2,musicldm,5521
3,mustango,5521
4,stable_audio_open,5521


Total files: 27605


In [5]:
records = []
for folder in generator_dirs:
    for file_path in sorted(folder.glob("*.wav")):
        records.append({
            "track_id": file_path.stem,
            "generator": folder.name,
            "path": str(file_path),
        })

audio_df = pd.DataFrame(records)
display(audio_df.head())
print("Dataset shape:", audio_df.shape)

,track_id,generator,path
0,-0Gj8-vB1q4,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0Gj8-vB1q...
1,-0SdAVK79lg,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0SdAVK79l...
2,-0vPFx-wRRI,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0vPFx-wRR...
3,-0xzrMun0Rs,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0xzrMun0R...
4,-1LrH01Ei1w,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-1LrH01Ei1...


Dataset shape: (27605, 3)


Track ID Consistency


In [6]:
track_coverage = (audio_df.groupby("track_id")["generator"].nunique())
display(track_coverage.value_counts().sort_index())

generator
5    5521
Name: count, dtype: int64

In [7]:
print("Missing values:")
display(audio_df.isna().sum())

print("Duplicate generator-track pairs:")
print(audio_df.duplicated(subset=["generator", "track_id"]).sum())

Missing values:


track_id     0
generator    0
path         0
dtype: int64

Duplicate generator-track pairs:
0


The dataset contains 27,605 audio files, evenly distributed across five generators. All 5,521 unique track IDs are represented by every generator. No missing values or duplicate `(generator, track_id)` pairs were found in the file manifest. This confirms that the dataset has a consistent file organization.

### 2. Metadata Exploration

This section examines the available metadata and the relationship between audio files, generators, and text captions.

In [8]:
METADATA_PATH = Path("../data/raw/musiccaps-public.csv")

if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Metadata not found: {METADATA_PATH}. Run scripts/download_data.py first.")
metadata_df = pd.read_csv(METADATA_PATH)
print("Metadata shape:", metadata_df.shape)
display(metadata_df.head())

Metadata shape: (5521, 9)


,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False


In [9]:
print("Columns:")
print(metadata_df.columns.tolist())
print("\nMissing values:")
display(metadata_df.isna().sum())
print("\nDuplicate IDs:")
print(metadata_df["ytid"].duplicated().sum())
print("\nUnique IDs:")
print(metadata_df["ytid"].nunique())

Columns:
['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval']

Missing values:


ytid                        0
start_s                     0
end_s                       0
audioset_positive_labels    0
aspect_list                 0
caption                     0
author_id                   0
is_balanced_subset          0
is_audioset_eval            0
dtype: int64


Duplicate IDs:
0

Unique IDs:
5521


In [10]:
audio_ids = set(audio_df["track_id"])
metadata_ids = set(metadata_df["ytid"])
print("Matching track IDs:", len(audio_ids & metadata_ids))
print("Unmatched audio IDs:", len(audio_ids - metadata_ids))

audio_metadata_df = audio_df.merge(
    metadata_df[["ytid", "caption"]],
    left_on="track_id",
    right_on="ytid",
    how="left",
    validate="many_to_one",
).drop(columns="ytid")

assert audio_metadata_df["caption"].notna().all()
display(audio_metadata_df.head())
print("Final shape:", audio_metadata_df.shape)

Matching track IDs: 5521
Unmatched audio IDs: 0


,track_id,generator,path,caption
0,-0Gj8-vB1q4,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0Gj8-vB1q...,The low quality recording features a ballad so...
1,-0SdAVK79lg,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0SdAVK79l...,This song features an electric guitar as the m...
2,-0vPFx-wRRI,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0vPFx-wRR...,a male voice is singing a melody with changing...
3,-0xzrMun0Rs,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0xzrMun0R...,This song contains digital drums playing a sim...
4,-1LrH01Ei1w,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-1LrH01Ei1...,This song features a rubber instrument being p...


Final shape: (27605, 4)


In [11]:
OUTPUT_PATH = Path("../data/interim/audio_metadata.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
audio_metadata_df.to_csv(OUTPUT_PATH, index=False)
print(f"Metadata saved to: {OUTPUT_PATH}")

Metadata saved to: ..\data\interim\audio_metadata.csv


### Metadata Exploration - Summary

FakeMusicCaps contains 27,605 audio files generated by five text-to-music models, with 5,521 files per generator. MusicCaps metadata was used to obtain the original text captions. All 5,521 track IDs were successfully matched, resulting in a dataset with four columns: `track_id`, `generator`, `path`, and `caption`. No missing values, duplicate generator-track pairs, or repeated captions across different track IDs were found.


## 3. Technical Audio Analysis

Now we analyze the technical characteristics of the audio signals. The analysis is performed on all 27,605 audio files using the previously constructed `audio_metadata_df`.
Examine:
- audio duration;
- sample rate;
- number of channels;
- file format;
- peak amplitude;
and RMS amplitude to identify potential differences between generators that may affect feature extraction and classification.

In [13]:
from ai_music_origin.data import load_audio

In [14]:
audio_stats = []
for row in audio_metadata_df.itertuples(index=False):
    audio, sr = load_audio(row.path)
    info = sf.info(row.path)
    audio_stats.append({
        "track_id": row.track_id,
        "generator": row.generator,
        "duration": len(audio) / sr,
        "sample_rate": sr,
        "channels": info.channels,
        "format": info.format,
        "subtype": info.subtype,
        "max_amplitude": np.max(np.abs(audio)),
        "rms": np.sqrt(np.mean(audio.astype(np.float64) ** 2)),
    })

stats_df = pd.DataFrame(audio_stats)
audio_analysis_df = audio_metadata_df.merge(stats_df, on=["track_id", "generator"], how="left", validate="one_to_one")
display(audio_analysis_df.head())
print("Shape:", audio_analysis_df.shape)


,track_id,generator,path,caption,duration,sample_rate,channels,format,subtype,max_amplitude,rms
0,-0Gj8-vB1q4,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0Gj8-vB1q...,The low quality recording features a ballad so...,10.0,16000,1,WAV,FLOAT,0.606934,0.132872
1,-0SdAVK79lg,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0SdAVK79l...,This song features an electric guitar as the m...,10.0,16000,1,WAV,FLOAT,0.492920,0.083085
2,-0vPFx-wRRI,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0vPFx-wRR...,a male voice is singing a melody with changing...,10.0,16000,1,WAV,FLOAT,0.685547,0.080026
3,-0xzrMun0Rs,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0xzrMun0R...,This song contains digital drums playing a sim...,10.0,16000,1,WAV,FLOAT,0.608887,0.102995
4,-1LrH01Ei1w,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-1LrH01Ei1...,This song features a rubber instrument being p...,10.0,16000,1,WAV,FLOAT,0.510742,0.097095


Shape: (27605, 11)


### Technical characteristics

In [25]:
technical_summary = audio_analysis_df.groupby("generator").agg(
    num_files=("track_id", "count"),
    duration=("duration", "mean"),
    sample_rate=("sample_rate", "first"),
    channels=("channels", "first"),
    format=("format", "first"),
    subtype=("subtype", "first"),
)

display(technical_summary.round(3))

,num_files,duration,sample_rate,channels,format,subtype
generator,,,,,,
MusicGen_medium,5521,10.180,16000,1,WAV,FLOAT
audioldm2,5521,10.000,16000,1,WAV,FLOAT
musicldm,5521,10.000,16000,1,WAV,FLOAT
mustango,5521,10.242,16000,1,WAV,FLOAT
stable_audio_open,5521,10.000,16000,1,WAV,FLOAT


All 27,605 audio files are mono, sampled at 16 kHz, and stored in WAV format with FLOAT encoding. Audio duration is consistent within each generator but differs between generators. These duration differences could introduce generator-specific shortcuts during classification. Therefore, audio clips should be adjusted to a consistent duration before feature extraction.

### Amplitude Analysis

In [19]:
amplitude_summary = audio_analysis_df.groupby("generator").agg(mean_peak_amplitude=("max_amplitude", "mean"),
                                                                max_peak_amplitude=("max_amplitude", "max"),
                                                                mean_rms=("rms", "mean"))
display(amplitude_summary.round(4))

,mean_peak_amplitude,max_peak_amplitude,mean_rms
generator,,,
MusicGen_medium,0.5751,1.7786,0.0985
audioldm2,0.5470,0.9937,0.0941
musicldm,0.5474,0.9941,0.0990
mustango,0.6997,1.0000,0.1196
stable_audio_open,0.7441,2.3065,0.1408


In [20]:
peak_above_one = (audio_analysis_df.assign(above_one=audio_analysis_df["max_amplitude"] > 1).groupby("generator")["above_one"].agg(["sum", "mean"]))
peak_above_one["percentage"] = peak_above_one["mean"] * 100
display(peak_above_one.round(2))

,sum,mean,percentage
generator,,,
MusicGen_medium,445,0.08,8.06
audioldm2,0,0.00,0.00
musicldm,0,0.00,0.00
mustango,0,0.00,0.00
stable_audio_open,1127,0.20,20.41


Amplitude characteristics differ across generators. Stable Audio Open has the highest mean peak amplitude (0.7441) and mean RMS amplitude (0.1408), while AudioLDM2 and MusicLDM show relatively lower average values. Peak amplitudes above 1 were observed in 445 MusicGen Medium files (8.06%) and 1,127 Stable Audio Open files (20.41%). Such values are possible in floating-point audio and do not necessarily indicate corrupted files. These differences may provide generator-specific information, but they could also reflect differences in audio scaling or storage. Their impact on classification should therefore be investigated in later experiments.

In [28]:

OUTPUT_PATH = Path("../data/interim/audio_analysis.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
audio_analysis_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(audio_analysis_df)} records to {OUTPUT_PATH}")


Saved 27605 records to ..\data\interim\audio_analysis.csv


### Preprocessing test

In [30]:
from ai_music_origin.data import preprocess_audio

In [31]:
test_file = next(generator_dirs[0].glob("*.wav"))
audio, sr = load_audio(test_file)

print("Original shape:", audio.shape)
print("Sample rate:", sr)
print("Original duration:", len(audio) / sr)

Original shape: (160000,)
Sample rate: 16000
Original duration: 10.0


In [32]:
processed = preprocess_audio(audio, sr, target_duration=10.0, normalize=False)
print("Processed shape:", processed.shape)
print("Processed duration:", len(processed) / sr)

Processed shape: (160000,)
Processed duration: 10.0


In [33]:
normalized = preprocess_audio(audio, sr, normalize=True)
print("Original max amplitude:", np.max(np.abs(audio)))
print("Normalized max amplitude:", np.max(np.abs(normalized)))

Original max amplitude: 0.6069336
Normalized max amplitude: 1.0


In [34]:
mustango_dir = DATA_DIR / "mustango"
test_file_long = next(mustango_dir.glob("*.wav"))
audio_long, sr_long = load_audio(test_file_long)
processed_long = preprocess_audio(audio_long, sr_long, target_duration=10.0, normalize=False)

print("Original duration:", len(audio_long) / sr_long)
print("Processed duration:", len(processed_long) / sr_long)

Original duration: 10.242
Processed duration: 10.0


In [ ]:
# Test zero-padding with a synthetic 5-second signal
short_audio = np.ones(5 * 16000, dtype=np.float32)
padded_audio = preprocess_audio(short_audio, sample_rate=16000, target_duration=10.0, normalize=False)
print("Original duration:", len(short_audio) / 16000)
print("Processed duration:", len(padded_audio) / 16000)

Original duration: 5.0
Processed duration: 10.0


The preprocessing functions work correctly on the tested audio files. Audio can be loaded without changing the original data, and optional preprocessing steps such as trimming, padding, and amplitude normalization can be applied when needed. These steps are not applied by default.